<a href="https://colab.research.google.com/github/rsekola/Projet_data/blob/master/Extraction_de__donn%C3%A9es__Web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import requests
from bs4 import BeautifulSoup

def recuperer_html(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        )
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"Erreur lors de la récupération : {response.status_code}")
    return BeautifulSoup(response.text, "html.parser")


In [13]:
def extraire_titre(soup):
    titre = soup.find("h1", {"id": "firstHeading"})
    return titre.text.strip() if titre else "Titre introuvable"

In [14]:
def extraire_contenu(soup):
    contenu = {}
    corps = soup.find("div", {"id": "bodyContent"})

    if not corps:
        return contenu

    section_courante = "Introduction"
    contenu[section_courante] = []

    for element in corps.find_all(["h2", "p"]):
        if element.name == "h2":
            span = element.find("span", {"class": "mw-headline"})
            if span:
                section_courante = span.text.strip()
                contenu[section_courante] = []
        elif element.name == "p":
            texte = element.get_text().strip()
            if texte:
                contenu[section_courante].append(texte)

    return contenu


In [15]:
from urllib.parse import urljoin

def extraire_liens(soup, base_url="https://fr.wikipedia.org"):
    liens = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/wiki/") and not any(x in href for x in [":", "#"]):
            lien_complet = urljoin(base_url, href)
            liens.add(lien_complet)
    return sorted(liens)


In [16]:
def extraire_infos_wikipedia(url):
    soup = recuperer_html(url)
    titre = extraire_titre(soup)
    contenu = extraire_contenu(soup)
    liens = extraire_liens(soup)

    return {
        "url": url,
        "titre": titre,
        "contenu": contenu,
        "liens_internes": liens
    }


In [17]:
if __name__ == "__main__":
    url_test = "https://fr.wikipedia.org/wiki/Python_(langage)"
    donnees = extraire_infos_wikipedia(url_test)

    print(f"\nTitre de l'article : {donnees['titre']}\n")
    print("Sections disponibles :")
    for section in donnees["contenu"].keys():
        print(f" - {section}")

    print(f"\nNombre de liens internes collectés : {len(donnees['liens_internes'])}")
    print(f"Exemple de quelques liens : {donnees['liens_internes'][:5]}")



Titre de l'article : Python (langage)

Sections disponibles :
 - Introduction

Nombre de liens internes collectés : 499
Exemple de quelques liens : ['https://fr.wikipedia.org/wiki/%C3%89vang%C3%A9liste_technologique', 'https://fr.wikipedia.org/wiki/13_septembre', 'https://fr.wikipedia.org/wiki/14_mars', 'https://fr.wikipedia.org/wiki/15_avril', 'https://fr.wikipedia.org/wiki/16_mars']
